In [1]:
!pip -q install pyspark
import time
import os
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, sum, avg, desc, to_timestamp,
    year, month, datediff, dayofweek, dense_rank,
    row_number, lag, spark_partition_id
)
from pyspark.sql.window import Window
from pyspark import StorageLevel

spark = SparkSession.builder.appName("StreamingSet4").getOrCreate()

def read_csv(name):
    return spark.read.csv(name, header=True, inferSchema=True)

files.upload()

sessions_df = read_csv("watch_sessions_80000.csv")
extra_df = read_csv("watch_sessions_extra_10000.csv")
subscribers_df = read_csv("subscriber_master_2400.csv")
content_df = read_csv("content_catalog_120.csv")
devices_df = read_csv("device_master_8.csv")
plans_df = read_csv("subscription_plans_4.csv")

Saving content_catalog_120.csv to content_catalog_120.csv
Saving device_master_8.csv to device_master_8.csv
Saving subscriber_master_2400.csv to subscriber_master_2400.csv
Saving subscription_plans_4.csv to subscription_plans_4.csv
Saving watch_sessions_80000.csv to watch_sessions_80000.csv
Saving watch_sessions_extra_10000.csv to watch_sessions_extra_10000.csv


**Part A - DataFrame operations**

In [2]:
# 01. Display the schema, row count, and first 10 records of sessions_df. Identify numeric, string, and timestamp fields.
sessions_df.printSchema()
print("Total row count:", sessions_df.count())
sessions_df.show(10)


root
 |-- session_id: string (nullable = true)
 |-- subscriber_id: string (nullable = true)
 |-- content_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- session_start: timestamp (nullable = true)
 |-- watch_minutes: integer (nullable = true)
 |-- completion_pct: double (nullable = true)
 |-- buffering_seconds: double (nullable = true)
 |-- stream_quality: string (nullable = true)
 |-- session_status: string (nullable = true)
 |-- data_used_mb: double (nullable = true)
 |-- user_rating: double (nullable = true)

Total row count: 80000
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+------------

In [3]:
# 02. Select session_id, subscriber_id, content_id, watch_minutes, and session_status. Filter completed sessions with watch_minutes greater than 60.
filtered_sessions = sessions_df.select(
    "session_id", "subscriber_id", "content_id", "watch_minutes", "session_status"
).filter(
    (col("session_status") == "completed") & (col("watch_minutes") > 60)
)

filtered_sessions.show(10)

+----------+-------------+----------+-------------+--------------+
|session_id|subscriber_id|content_id|watch_minutes|session_status|
+----------+-------------+----------+-------------+--------------+
+----------+-------------+----------+-------------+--------------+



In [4]:
# 03. Create watch_hours from watch_minutes and a high_engagement flag for sessions with at least 75% completion. Preserve null completion values.
engagement_df = sessions_df.withColumn(
    "watch_hours", col("watch_minutes") / 60
).withColumn(
    "high_engagement",
    when(col("completion_pct").isNull(), None)
    .when(col("completion_pct") >= 75, 1)
    .otherwise(0)
)

engagement_df.select("session_id", "watch_minutes", "watch_hours", "completion_pct", "high_engagement").show(10)

+----------+-------------+-------------------+--------------+---------------+
|session_id|watch_minutes|        watch_hours|completion_pct|high_engagement|
+----------+-------------+-------------------+--------------+---------------+
|WS00000001|            6|                0.1|          11.8|              0|
|WS00000002|           31| 0.5166666666666667|          56.4|              0|
|WS00000003|           29|0.48333333333333334|          63.0|              0|
|WS00000004|           69|               1.15|          75.0|              1|
|WS00000005|           36|                0.6|          23.4|              0|
|WS00000006|           20| 0.3333333333333333|          19.8|              0|
|WS00000007|           78|                1.3|          83.9|              1|
|WS00000008|           22|0.36666666666666664|          66.7|              0|
|WS00000009|           21|               0.35|          80.8|              1|
|WS00000010|           21|               0.35|          65.6|   

In [5]:
# 04. Sort sessions by data_used_mb descending and display the top 20 records.
sorted_sessions = sessions_df.orderBy(col("data_used_mb").desc())
sorted_sessions.show(20)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00051169|     SUB00155|    CT0069|     DV08|2026-07-20 00:11:00|          167|          97.7|              9.0|            4K|     Completed|      8881.8|        3.0|
|WS00074513|     SUB00684|    CT0056|     DV05|2026-08-09 21:53:00|          157|          96.9|              8.0|            4K|     Completed|      8408.4|       NULL|
|WS00039201|     SUB01664|    CT0027|     DV08|2026-05-16 06:35:00|          161|          97.0|              9.0|            4K|     Completed|      

**Part B - Data quality and conditional transformations**

In [6]:
# 05. Count missing values in subscriber_id, device_id, completion_pct, buffering_seconds, and user_rating.
null_counts_df = sessions_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["subscriber_id", "device_id", "completion_pct", "buffering_seconds", "user_rating"]
])

null_counts_df.show()

+-------------+---------+--------------+-----------------+-----------+
|subscriber_id|device_id|completion_pct|buffering_seconds|user_rating|
+-------------+---------+--------------+-----------------+-----------+
|          951|     2003|          3513|             4424|      77605|
+-------------+---------+--------------+-----------------+-----------+



In [7]:
# 06. Create a new DataFrame that replaces missing buffering_seconds with zero. Do not modify the original DataFrame.
filled_buffering_df = sessions_df.fillna({"buffering_seconds": 0})
filled_buffering_df.select("session_id", "buffering_seconds").show(10)

+----------+-----------------+
|session_id|buffering_seconds|
+----------+-----------------+
|WS00000001|             29.0|
|WS00000002|             31.0|
|WS00000003|             26.0|
|WS00000004|              2.0|
|WS00000005|             31.0|
|WS00000006|            202.0|
|WS00000007|             28.0|
|WS00000008|             11.0|
|WS00000009|             12.0|
|WS00000010|            166.0|
+----------+-----------------+
only showing top 10 rows


In [9]:
# 07. Create another DataFrame that excludes sessions with missing device_id. Compare record counts.
valid_device_df = sessions_df.filter(col("device_id").isNotNull())

print("Original sessions count:", sessions_df.count())
print("Sessions with valid device_id count:", valid_device_df.count())

Original sessions count: 80000
Sessions with valid device_id count: 77997


In [10]:
# 08. Use when() and otherwise() to label completion_pct as Low (below 30), Medium (30 to below 90), High (90 or more), or Unknown (null).
labeled_completion_df = sessions_df.withColumn(
    "completion_label",
    when(col("completion_pct").isNull(), "Unknown")
    .when(col("completion_pct") < 30, "Low")
    .when((col("completion_pct") >= 30) & (col("completion_pct") < 90), "Medium")
    .when(col("completion_pct") >= 90, "High")
    .otherwise("Unknown")
)

labeled_completion_df.select("session_id", "completion_pct", "completion_label").show(10)

+----------+--------------+----------------+
|session_id|completion_pct|completion_label|
+----------+--------------+----------------+
|WS00000001|          11.8|             Low|
|WS00000002|          56.4|          Medium|
|WS00000003|          63.0|          Medium|
|WS00000004|          75.0|          Medium|
|WS00000005|          23.4|             Low|
|WS00000006|          19.8|             Low|
|WS00000007|          83.9|          Medium|
|WS00000008|          66.7|          Medium|
|WS00000009|          80.8|          Medium|
|WS00000010|          65.6|          Medium|
+----------+--------------+----------------+
only showing top 10 rows


**Part C - Joins and aggregations**

In [11]:
# 09. Join sessions with the content catalog and calculate session counts and total watch_minutes by genre.
genre_summary_df = sessions_df.join(content_df, on="content_id", how="inner") \
    .groupBy("genre") \
    .agg(
        count("session_id").alias("session_count"),
        sum("watch_minutes").alias("total_watch_minutes")
    )

genre_summary_df.show()

+-----------+-------------+-------------------+
|      genre|session_count|total_watch_minutes|
+-----------+-------------+-------------------+
|   Thriller|         5972|             248430|
|  Adventure|        15561|             839700|
|     Sports|         9967|             383044|
|      Drama|        10733|             519872|
|Documentary|         7432|             256626|
|  Animation|        11886|             490335|
|     Comedy|         9105|             452563|
|     Sci-Fi|         9344|             433946|
+-----------+-------------+-------------------+



In [12]:
# 10. Join sessions with the device master and calculate average buffering_seconds and average watch_minutes by device_type.
device_summary_df = sessions_df.join(devices_df, on="device_id", how="inner") \
    .groupBy("device_type") \
    .agg(
        avg("buffering_seconds").alias("avg_buffering_seconds"),
        avg("watch_minutes").alias("avg_watch_minutes")
    )

device_summary_df.show()

+---------------+---------------------+------------------+
|    device_type|avg_buffering_seconds| avg_watch_minutes|
+---------------+---------------------+------------------+
|         iPhone|   35.761106907361366|45.226752503576535|
|Streaming Stick|   36.320860309448044| 45.08570541351827|
|Desktop Browser|    36.02889125799574|45.571818733562615|
|  Android Phone|   35.676486693680864|  45.0885616509385|
| Laptop Browser|   36.930938301907844| 45.84465713083719|
|         Tablet|    35.23614616481949| 45.38943894389439|
|       Smart TV|    36.67392739273927| 45.33776623376623|
|   Game Console|    35.51243726816496| 45.04469158686026|
+---------------+---------------------+------------------+



In [13]:
# 11. Join sessions, subscribers, and plans. Calculate completion rate (completed sessions / all sessions) by plan_name, excluding null subscriber IDs.
plan_completion_df = sessions_df.filter(col("subscriber_id").isNotNull()) \
    .join(subscribers_df, on="subscriber_id", how="inner") \
    .join(plans_df, on="plan_id", how="inner") \
    .groupBy("plan_name") \
    .agg(
        (sum(when(col("session_status") == "completed", 1).otherwise(0)) / count("session_id")).alias("completion_rate")
    )

plan_completion_df.show()

+---------+---------------+
|plan_name|completion_rate|
+---------+---------------+
|  Premium|            0.0|
|    Basic|            0.0|
|   Mobile|            0.0|
| Standard|            0.0|
+---------+---------------+



In [15]:
subscribers_df.show()

+-------------+-----------+------+-------+---------+-----------------+
|subscriber_id|signup_date|region|plan_id|age_group|subscriber_status|
+-------------+-----------+------+-------+---------+-----------------+
|     SUB00001| 2025-06-17| South|   PL02|    35-44|           Active|
|     SUB00002| 2024-11-22| South|   PL02|    25-34|           Active|
|     SUB00003| 2024-02-10|  West|   PL03|    45-54|           Active|
|     SUB00004| 2025-03-05| North|   PL02|    45-54|           Active|
|     SUB00005| 2024-05-25| South|   PL01|    35-44|           Active|
|     SUB00006| 2024-05-17| South|   PL03|    45-54|           Active|
|     SUB00007| 2024-06-29| North|   PL02|    25-34|           Active|
|     SUB00008| 2024-11-28| South|   PL03|    35-44|           Active|
|     SUB00009| 2025-03-28| North|   PL03|    18-24|           Active|
|     SUB00010| 2025-11-16| North|   PL02|    25-34|           Active|
|     SUB00011| 2024-03-09| South|   PL03|    25-34|           Active|
|     

In [17]:
# 12. Using subscriber_master and subscription_plans only, estimate monthly recurring revenue by plan for Active subscribers. Count each subscriber once; do not multiply fees by viewing sessions.
mrr_by_plan_df = subscribers_df.filter(col("subscriber_status") == "Active") \
    .join(plans_df, on="plan_id", how="inner") \
    .groupBy("plan_name", "monthly_fee_inr") \
    .agg(count("subscriber_id").alias("active_subscribers")) \
    .withColumn("monthly_recurring_revenue", col("active_subscribers") * col("monthly_fee_inr"))

mrr_by_plan_df.show()

+---------+---------------+------------------+-------------------------+
|plan_name|monthly_fee_inr|active_subscribers|monthly_recurring_revenue|
+---------+---------------+------------------+-------------------------+
|   Mobile|            149|               443|                    66007|
|    Basic|            249|               632|                   157368|
|  Premium|            649|               326|                   211574|
| Standard|            399|               604|                   240996|
+---------+---------------+------------------+-------------------------+



In [18]:
# 13. Identify sessions whose non-null subscriber_id or device_id has no matching master record. Report unmatched counts for each key.
unmatched_subscribers = sessions_df.filter(col("subscriber_id").isNotNull()) \
    .join(subscribers_df, on="subscriber_id", how="left_anti") \
    .count()

unmatched_devices = sessions_df.filter(col("device_id").isNotNull()) \
    .join(devices_df, on="device_id", how="left_anti") \
    .count()

print("Sessions with unmatched subscriber_id:", unmatched_subscribers)
print("Sessions with unmatched device_id:", unmatched_devices)

Sessions with unmatched subscriber_id: 571
Sessions with unmatched device_id: 481


**Part D - Date and window functions**

In [19]:
# 14. Convert session_start to timestamp. Calculate monthly session counts and total watch_minutes for 2026.
monthly_2026_df = sessions_df.withColumn("session_timestamp", to_timestamp(col("session_start"))) \
    .filter(year(col("session_timestamp")) == 2026) \
    .groupBy(month(col("session_timestamp")).alias("month")) \
    .agg(
        count("session_id").alias("session_count"),
        sum("watch_minutes").alias("total_watch_minutes")
    ) \
    .orderBy("month")

monthly_2026_df.show()

+-----+-------------+-------------------+
|month|session_count|total_watch_minutes|
+-----+-------------+-------------------+
|    1|         9070|             407215|
|    2|         8218|             372726|
|    3|         9019|             403971|
|    4|         8817|             399432|
|    5|         9006|             407386|
|    6|         8776|             404203|
|    7|         9204|             416104|
|    8|         8996|             407374|
|    9|         8894|             406105|
+-----+-------------+-------------------+



In [20]:
# 15. Join subscriber signup dates and calculate days between signup_date and each viewing session. Display the earliest 10 viewing sessions.
days_since_signup_df = sessions_df.withColumn("session_timestamp", to_timestamp(col("session_start"))) \
    .join(subscribers_df, on="subscriber_id", how="inner") \
    .withColumn("days_since_signup", datediff(col("session_timestamp"), to_timestamp(col("signup_date")))) \
    .orderBy("session_timestamp")

days_since_signup_df.select("session_id", "subscriber_id", "signup_date", "session_start", "days_since_signup").show(10)

+----------+-------------+-----------+-------------------+-----------------+
|session_id|subscriber_id|signup_date|      session_start|days_since_signup|
+----------+-------------+-----------+-------------------+-----------------+
|WS00035235|     SUB01707| 2024-02-13|2026-01-01 00:00:00|              688|
|WS00074739|     SUB02206| 2025-08-31|2026-01-01 00:04:00|              123|
|WS00032058|     SUB01248| 2024-03-17|2026-01-01 00:04:00|              655|
|WS00048560|     SUB02342| 2025-08-25|2026-01-01 00:14:00|              129|
|WS00067087|     SUB00563| 2024-08-11|2026-01-01 00:15:00|              508|
|WS00004245|     SUB00711| 2024-10-26|2026-01-01 00:18:00|              432|
|WS00041710|     SUB00261| 2025-05-15|2026-01-01 00:21:00|              231|
|WS00045132|     SUB02167| 2025-11-22|2026-01-01 00:28:00|               40|
|WS00036438|     SUB01237| 2024-02-01|2026-01-01 00:41:00|              700|
|WS00030050|     SUB00646| 2024-11-20|2026-01-01 00:51:00|              407|

In [21]:
# 16. Classify sessions as Weekday or Weekend using the session date, then compare average watch_minutes.
weekday_weekend_df = sessions_df.withColumn("session_timestamp", to_timestamp(col("session_start"))) \
    .withColumn(
        "day_type",
        when(dayofweek(col("session_timestamp")).isin(1, 7), "Weekend")
        .otherwise("Weekday")
    ) \
    .groupBy("day_type") \
    .agg(avg("watch_minutes").alias("avg_watch_minutes"))

weekday_weekend_df.show()

+--------+------------------+
|day_type| avg_watch_minutes|
+--------+------------------+
| Weekday| 45.26460823373174|
| Weekend|45.411601967328295|
+--------+------------------+



In [23]:
# 17. Aggregate total watch_minutes by genre and content_title; use dense_rank() to identify the top three titles within each genre.
genre_title_watch = sessions_df.join(content_df, on="content_id", how="inner") \
    .groupBy("genre", "content_title") \
    .agg(sum("watch_minutes").alias("total_watch_minutes"))

genre_window = Window.partitionBy("genre").orderBy(col("total_watch_minutes").desc())

top_titles_df = genre_title_watch.withColumn("rank", dense_rank().over(genre_window)) \
    .filter(col("rank") <= 3)

top_titles_df.show(30)

+-----------+----------------+-------------------+----+
|      genre|   content_title|total_watch_minutes|rank|
+-----------+----------------+-------------------+----+
|  Adventure|Infinite Horizon|              66395|   1|
|  Adventure|    Lost Journey|              61823|   2|
|  Adventure|    Solar Memory|              60540|   3|
|  Animation|      Last River|              51398|   1|
|  Animation| Emerald Circuit|              51310|   2|
|  Animation|   Golden Puzzle|              50515|   3|
|     Comedy|    Blue Horizon|              61428|   1|
|     Comedy|   Bright Valley|              60921|   2|
|     Comedy|   Silver Valley|              60422|   3|
|Documentary|   Fading Signal|              47085|   1|
|Documentary|  Emerald Valley|              40819|   2|
|Documentary|   Solar Horizon|              36152|   3|
|      Drama|  Crimson Summit|              65724|   1|
|      Drama|  Rising Archive|              57301|   2|
|      Drama|     Wild Harbor|              4547

In [24]:
# 18. Use row_number() to retrieve the most recent session for each non-null subscriber_id.
subscriber_window = Window.partitionBy("subscriber_id").orderBy(col("session_start").desc(), col("session_id").desc())

latest_session_df = sessions_df.filter(col("subscriber_id").isNotNull()) \
    .withColumn("rn", row_number().over(subscriber_window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

latest_session_df.show(10)

+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00020822|     SUB00001|    CT0073|     DV03|2026-09-11 19:19:00|           34|          89.5|            145.0|            HD|   Interrupted|       418.3|       NULL|
|WS00032744|     SUB00002|    CT0076|     DV05|2026-09-02 05:41:00|           45|          73.8|            219.0|       Full HD|   Interrupted|      1184.1|       NULL|
|WS00006110|     SUB00003|    CT0004|     DV03|2026-09-30 03:13:00|           11|          15.3|              3.0|            4K|     Abandoned|      

In [25]:
# 19. Use lag() to compare each subscriber's watch_minutes with their previous session, ordered by session_start and session_id.
sub_timeline_window = Window.partitionBy("subscriber_id").orderBy(col("session_start"), col("session_id"))

lag_sessions_df = sessions_df.filter(col("subscriber_id").isNotNull()) \
    .withColumn("previous_watch_minutes", lag("watch_minutes", 1).over(sub_timeline_window)) \
    .withColumn("difference_minutes", col("watch_minutes") - col("previous_watch_minutes"))

lag_sessions_df.select("subscriber_id", "session_id", "session_start", "watch_minutes", "previous_watch_minutes", "difference_minutes").show(15)

+-------------+----------+-------------------+-------------+----------------------+------------------+
|subscriber_id|session_id|      session_start|watch_minutes|previous_watch_minutes|difference_minutes|
+-------------+----------+-------------------+-------------+----------------------+------------------+
|     SUB00001|WS00064390|2026-01-02 17:32:00|           58|                  NULL|              NULL|
|     SUB00001|WS00045183|2026-01-05 04:51:00|           78|                    58|                20|
|     SUB00001|WS00028484|2026-01-09 08:46:00|            4|                    78|               -74|
|     SUB00001|WS00043555|2026-01-24 19:41:00|          108|                     4|               104|
|     SUB00001|WS00041862|2026-02-18 07:43:00|           79|                   108|               -29|
|     SUB00001|WS00059616|2026-02-25 07:38:00|           18|                    79|               -61|
|     SUB00001|WS00042345|2026-03-13 07:34:00|           44|             

**Part E - Union, lazy execution, caching and partitioning**

In [26]:
# 20. Combine both session batches using unionByName(). Count rows and remove duplicate session_id values. Compare the totals.
unioned_df = sessions_df.unionByName(extra_df)
unioned_count = unioned_df.count()

deduped_df = unioned_df.dropDuplicates(["session_id"])
deduped_count = deduped_df.count()

print("Row count after unionByName:", unioned_count)
print("Row count after deduplication:", deduped_count)

Row count after unionByName: 90000
Row count after deduplication: 89400


In [27]:
# 21. Build a lazy pipeline with filter(), join(), withColumn(), groupBy(), and orderBy(). Inspect explain() before calling an action.
lazy_pipeline_df = sessions_df.filter(col("session_status") == "completed") \
    .join(content_df, on="content_id", how="inner") \
    .withColumn("watch_hours", col("watch_minutes") / 60) \
    .groupBy("genre") \
    .agg(avg("watch_hours").alias("avg_watch_hours")) \
    .orderBy("avg_watch_hours")

lazy_pipeline_df.explain(True)

== Parsed Logical Plan ==
'Sort ['avg_watch_hours ASC NULLS FIRST], true
+- Aggregate [genre#100], [genre#100, avg(watch_hours#1038) AS avg_watch_hours#1039]
   +- Project [content_id#19, session_id#17, subscriber_id#18, device_id#20, session_start#21, watch_minutes#22, completion_pct#23, buffering_seconds#24, stream_quality#25, session_status#26, data_used_mb#27, user_rating#28, content_title#99, genre#100, content_type#101, duration_minutes#102, release_year#103, content_language#104, (cast(watch_minutes#22 as double) / cast(60 as double)) AS watch_hours#1038]
      +- Project [content_id#19, session_id#17, subscriber_id#18, device_id#20, session_start#21, watch_minutes#22, completion_pct#23, buffering_seconds#24, stream_quality#25, session_status#26, data_used_mb#27, user_rating#28, content_title#99, genre#100, content_type#101, duration_minutes#102, release_year#103, content_language#104]
         +- Join Inner, (content_id#19 = content_id#98)
            :- Filter (session_status#

In [28]:
# 22. Cache completed sessions, run count() and an aggregation twice, and compare timings. Explain when the cache is populated.
completed_cached_df = sessions_df.filter(col("session_status") == "completed").cache()
t0 = time.time()
print("Run 1 Count:", completed_cached_df.count())
completed_cached_df.agg(avg("watch_minutes")).show()
print("Run 1 Elapsed Time:", round(time.time() - t0, 4), "seconds")

t0 = time.time()
print("Run 2 Count:", completed_cached_df.count())
completed_cached_df.agg(avg("watch_minutes")).show()
print("Run 2 Elapsed Time:", round(time.time() - t0, 4), "seconds")

completed_cached_df.unpersist()

Run 1 Count: 0
+------------------+
|avg(watch_minutes)|
+------------------+
|              NULL|
+------------------+

Run 1 Elapsed Time: 1.2262 seconds
Run 2 Count: 0
+------------------+
|avg(watch_minutes)|
+------------------+
|              NULL|
+------------------+

Run 2 Elapsed Time: 0.2503 seconds


DataFrame[session_id: string, subscriber_id: string, content_id: string, device_id: string, session_start: timestamp, watch_minutes: int, completion_pct: double, buffering_seconds: double, stream_quality: string, session_status: string, data_used_mb: double, user_rating: double]

In [30]:
# 23. Persist sessions with stream_quality = "4K" using MEMORY_AND_DISK. Execute two actions and then call unpersist().
persisted_4k_df = sessions_df.filter(col("stream_quality") == "4K").persist(StorageLevel.MEMORY_AND_DISK)

print("Action 1 (count):", persisted_4k_df.count())
print("Action 2 :")
persisted_4k_df.show(5)

persisted_4k_df.unpersist()
print("Unpersisted successfully.")

Action 1 (count): 7312
Action 2 :
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|session_id|subscriber_id|content_id|device_id|      session_start|watch_minutes|completion_pct|buffering_seconds|stream_quality|session_status|data_used_mb|user_rating|
+----------+-------------+----------+---------+-------------------+-------------+--------------+-----------------+--------------+--------------+------------+-----------+
|WS00000005|     SUB01268|    CT0052|     DV08|2026-04-06 04:33:00|           36|          23.4|             31.0|            4K|     Abandoned|      1561.1|       NULL|
|WS00000010|     SUB00947|    CT0063|     DV02|2026-08-09 09:54:00|           21|          65.6|            166.0|            4K|   Interrupted|      1065.3|       NULL|
|WS00000017|     SUB02006|    CT0027|     DV04|2026-01-15 14:05:00|          140|          84.3|             12.0|  

In [31]:
# 24. Repartition sessions into 8 partitions and display row counts per partition using spark_partition_id().
repartitioned_df = sessions_df.repartition(8)

repartitioned_df.groupBy(spark_partition_id().alias("partition_id")) \
    .agg(count("*").alias("row_count")) \
    .orderBy("partition_id") \
    .show()

+------------+---------+
|partition_id|row_count|
+------------+---------+
|           0|    10000|
|           1|     9999|
|           2|    10001|
|           3|    10000|
|           4|    10000|
|           5|    10000|
|           6|    10000|
|           7|    10000|
+------------+---------+



In [32]:
# 25. Starting from 8 partitions, compare coalesce(2) and repartition(2) with explain(). Write a chosen output DataFrame to Parquet and inspect the part files.
base_df = sessions_df.repartition(8)

coalesced_df = base_df.coalesce(2)
repartitioned_2_df = base_df.repartition(2)

print("coalesce(2) plan")
coalesced_df.explain()

print("\nrepartition(2) plan ")
repartitioned_2_df.explain()

# Write coalesce(2) output to Parquet
output_path = "output_sessions_parquet"
coalesced_df.write.mode("overwrite").parquet(output_path)

print("\nGenerated files in Parquet output:")
for f in os.listdir(output_path):
    if f.endswith(".parquet") or f.startswith("part-"):
        print(f)

coalesce(2) plan
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 2
   +- Exchange RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=2677]
      +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_status#26,data_used_mb#27,user_rating#28] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/watch_sessions_80000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<session_id:string,subscriber_id:string,content_id:string,device_id:string,session_start:ti...



repartition(2) plan 
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=2686]
   +- FileScan csv [session_id#17,subscriber_id#18,content_id#19,device_id#20,session_start#21,watch_minutes#22,completion_pct#23,buffering_seconds#24,stream_quality#25,session_statu

**Final capstone - Streaming performance report**

In [34]:
# Final capstone - Streaming performance report

# 1. Union the two batches and deduplicate by session_id.
raw_union_df = sessions_df.unionByName(extra_df)
clean_sessions_df = raw_union_df.dropDuplicates(["session_id"])

# 2. Clean relevant nulls without discarding every incomplete record.
clean_sessions_df = clean_sessions_df.fillna({"buffering_seconds": 0}) \
    .filter(col("subscriber_id").isNotNull() & col("content_id").isNotNull())

# 3. Join subscribers, content, devices, and plans as needed.
joined_capstone_df = clean_sessions_df \
    .join(subscribers_df, on="subscriber_id", how="inner") \
    .join(content_df, on="content_id", how="inner")

# 4. Produce monthly metrics by region and genre: session count, watch hours, completion rate, average buffering, and total data usage.
monthly_report_df = joined_capstone_df.withColumn("session_timestamp", to_timestamp(col("session_start"))) \
    .withColumn("month", month(col("session_timestamp"))) \
    .groupBy("region", "genre", "month") \
    .agg(
        count("session_id").alias("session_count"),
        (sum("watch_minutes") / 60).alias("total_watch_hours"),
        (sum(when(col("session_status") == "completed", 1).otherwise(0)) / count("session_id")).alias("completion_rate"),
        avg("buffering_seconds").alias("avg_buffering_seconds"),
        sum("data_used_mb").alias("total_data_used_mb")
    )

# 5. Rank the top three content titles by watch hours within each genre.
genre_title_hours = joined_capstone_df.groupBy("genre", "content_title") \
    .agg((sum("watch_minutes") / 60).alias("title_watch_hours"))

capstone_window = Window.partitionBy("genre").orderBy(col("title_watch_hours").desc())
top_titles_report = genre_title_hours.withColumn("rank", dense_rank().over(capstone_window)) \
    .filter(col("rank") <= 3)

print("Top 3 Titles within each Genre:")
top_titles_report.show(20)

# 6. Cache the final aggregated DataFrame, reuse it for two actions, and unpersist it.
cached_report_df = monthly_report_df.cache()

print("Action 1 - Row count:", cached_report_df.count())
print("Action 2 - Sample data:")
cached_report_df.show(10)

cached_report_df.unpersist()

# 7. Save the final report as Parquet with 3 output partitions.
report_output_dir = "final_streaming_report_parquet"
monthly_report_df.coalesce(3).write.mode("overwrite").parquet(report_output_dir)

print("\nFiles in Parquet output directory:")
for file_name in os.listdir(report_output_dir):
    if file_name.endswith(".parquet") or file_name.startswith("part-"):
        print(file_name)

Top 3 Titles within each Genre:
+-----------+----------------+------------------+----+
|      genre|   content_title| title_watch_hours|rank|
+-----------+----------------+------------------+----+
|  Adventure|Infinite Horizon|1192.7333333333333|   1|
|  Adventure|    Lost Journey|           1108.05|   2|
|  Adventure|    Solar Memory|1104.9833333333333|   3|
|  Animation|      Last River| 956.6333333333333|   1|
|  Animation| Emerald Circuit| 935.5666666666667|   2|
|  Animation|   Golden Puzzle|             916.0|   3|
|     Comedy|   Bright Valley|           1156.05|   1|
|     Comedy|    Blue Horizon|1129.9833333333333|   2|
|     Comedy|     Wild Canvas|           1118.95|   3|
|Documentary|   Fading Signal| 867.7166666666667|   1|
|Documentary|  Emerald Valley| 730.9333333333333|   2|
|Documentary|   Solar Horizon| 673.8666666666667|   3|
|      Drama|  Crimson Summit|1210.6166666666666|   1|
|      Drama|  Rising Archive|1053.7833333333333|   2|
|      Drama|     Wild Harbor|   